In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 1 ── Import Libraries
# ─────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

print('Libraries imported successfully!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2 ── Settings
# ─────────────────────────────────────────────────────────────
# VGG-16 was designed for 224 x 224 images.
# We use a small batch size so it fits in memory.

IMG_SIZE   = (224, 224)   # width and height of each image
BATCH_SIZE = 32           # number of images processed at once
EPOCHS     = 25           # how many times we go through the full dataset

TRAIN_PATH = 'dataset/training_set'   # folder with cats/ and dogs/ subfolders
TEST_PATH  = 'dataset/test_set'       # folder with cats/ and dogs/ subfolders

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 3 ── Load & Prepare the Data
# ─────────────────────────────────────────────────────────────
# ImageDataGenerator does two things:
#   1. Rescale pixel values from 0-255  →  0-1  (makes training easier)
#   2. Augment training images so the model sees more variety
#      (flip, zoom, shear) — helps prevent overfitting

# --- Training data: rescale + augmentation ---
train_datagen = ImageDataGenerator(
    rescale         = 1.0 / 255,   # pixel values: 0-255  →  0-1
    horizontal_flip = True,         # randomly flip images left-right
    zoom_range      = 0.2,          # randomly zoom in up to 20%
    shear_range     = 0.2           # slight tilt/shear
)

# --- Test data: only rescale (no augmentation on test!) ---
test_datagen = ImageDataGenerator(
    rescale = 1.0 / 255
)

# Load training images from the folder.
# Keras reads the subfolder names (cats, dogs) as class labels automatically.
train_set = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary'   # binary = 2 classes (cat=0, dog=1)
)

# Load test images
test_set = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary'
)

# Show which label number belongs to which animal
print('Class labels:', train_set.class_indices)  # {'cats': 0, 'dogs': 1}

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4 ── Preview Some Training Images
# ─────────────────────────────────────────────────────────────

# Grab one batch of images and labels
images, labels = next(train_set)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i])                     # image is already 0-1 scale
    title = 'Dog' if labels[i] == 1 else 'Cat'
    plt.title(title, fontsize=11)
    plt.axis('off')

plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5 ── Build the VGG-16 Model
# ─────────────────────────────────────────────────────────────
#
# VGG-16 Architecture:
#
#  INPUT IMAGE (224 x 224 x 3)
#  │
#  ├── BLOCK 1 : Conv(64)  → Conv(64)  → MaxPool   →  112 x 112
#  ├── BLOCK 2 : Conv(128) → Conv(128) → MaxPool   →   56 x  56
#  ├── BLOCK 3 : Conv(256) x3 → MaxPool            →   28 x  28
#  ├── BLOCK 4 : Conv(512) x3 → MaxPool            →   14 x  14
#  ├── BLOCK 5 : Conv(512) x3 → MaxPool            →    7 x   7
#  │
#  ├── Flatten                                      →  25,088
#  ├── Dense(4096, relu) + Dropout(0.5)
#  ├── Dense(4096, relu) + Dropout(0.5)
#  └── Dense(1, sigmoid)  ←  0 = cat,  1 = dog
#
# Rules that NEVER change in VGG-16:
#   Every Conv  →  3x3 filter, padding='same', activation='relu'
#   Every Pool  →  2x2, strides=2  (cuts image size in half)
# ─────────────────────────────────────────────────────────────

model = Sequential()

# ── Block 1 ──────────────────────────────────────────────────
model.add(Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))   # 224x224 → 112x112

# ── Block 2 ──────────────────────────────────────────────────
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))   # 112x112 → 56x56

# ── Block 3 ──────────────────────────────────────────────────
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))   # 56x56 → 28x28

# ── Block 4 ──────────────────────────────────────────────────
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))   # 28x28 → 14x14

# ── Block 5 ──────────────────────────────────────────────────
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))   # 14x14 → 7x7

# ── Classifier Head ──────────────────────────────────────────
model.add(Flatten())                    # turn 7x7x512 → a list of 25,088 numbers

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))                 # turn off 50% of neurons randomly → reduces overfitting

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))   # outputs 0 (cat) or 1 (dog)

# Print every layer with its output shape and parameter count
model.summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6 ── Compile the Model
# ─────────────────────────────────────────────────────────────
# optimizer : Adam adjusts weights automatically during training
# loss      : binary_crossentropy is used when we have 2 classes
# metrics   : accuracy tells us how many images we classify correctly

model.compile(
    optimizer = 'adam',
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

print('Model compiled!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 7 ── Train the Model
# ─────────────────────────────────────────────────────────────
# model.fit() runs the training loop.
#   train_set       → images the model learns from
#   validation_data → images used to check progress (model never learns from these)
#   epochs          → how many full passes through all training images
#
# NOTE: VGG-16 is a large model.
#       If you have no GPU, set EPOCHS = 5 just to verify it runs.

history = model.fit(
    train_set,
    epochs          = EPOCHS,
    validation_data = test_set
)

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8 ── Plot Training vs Validation Accuracy & Loss
# ─────────────────────────────────────────────────────────────
# A well-trained model shows:
#   Accuracy going UP   over epochs
#   Loss going DOWN     over epochs
#   Training and Validation lines staying close (not diverging)

acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs   = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# ── Accuracy plot ──
plt.subplot(1, 2, 1)
plt.plot(epochs, acc,     'b-o', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r-o', label='Validation Accuracy')
plt.title('Training vs Validation Accuracy', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# ── Loss plot ──
plt.subplot(1, 2, 2)
plt.plot(epochs, loss,     'b-o', label='Training Loss')
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss')
plt.title('Training vs Validation Loss', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.suptitle('VGG-16 ── Cats & Dogs', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 9 ── Evaluate on the Test Set
# ─────────────────────────────────────────────────────────────

test_loss, test_acc = model.evaluate(test_set)

print(f'Test Accuracy : {test_acc * 100:.2f} %')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 10 ── Predict a Single Image
# ─────────────────────────────────────────────────────────────
# Put any image you want to test inside:
#   dataset/single_prediction/
# Then change IMAGE_NAME below to match your file.

import os

# ── Change this to the image you want to predict ──
IMAGE_NAME = 'cat_or_dog_1.jpg'

image_path = os.path.join('dataset', 'single_prediction', IMAGE_NAME)

# Load and resize the image to 224x224 (same size the model expects)
img = load_img(image_path, target_size=(224, 224))

# Convert image to a numpy array  →  shape: (224, 224, 3)
img_array = img_to_array(img)

# Rescale pixels 0-255 → 0-1  (same as we did during training)
img_array = img_array / 255.0

# Add a batch dimension: (224, 224, 3)  →  (1, 224, 224, 3)
# The model always expects a batch, even for a single image
img_array = np.expand_dims(img_array, axis=0)

# Make the prediction — returns a value between 0 and 1
prediction = model.predict(img_array)[0][0]

# Decide: above 0.5 = Dog, below 0.5 = Cat
if prediction > 0.5:
    result     = 'Dog'
    confidence = prediction * 100
else:
    result     = 'Cat'
    confidence = (1 - prediction) * 100

# Show the image with the result
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.axis('off')
plt.title(f'Prediction : {result}\nConfidence : {confidence:.1f}%', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Result     : {result}')
print(f'Confidence : {confidence:.1f}%')
print(f'Raw output : {prediction:.4f}  (0.0 = cat  |  1.0 = dog)')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 11 ── Save the Model
# ─────────────────────────────────────────────────────────────
# Saving lets you reload the trained model later without retraining.

model.save('vgg16_cats_dogs.h5')
print('Model saved as  vgg16_cats_dogs.h5')

# ── To reload it next time ──
# from tensorflow.keras.models import load_model
# model = load_model('vgg16_cats_dogs.h5')